# Text to Speech — Azure AI Speech

This notebook synthesizes text into natural-sounding speech and saves the output to a WAV file.

Azure AI Speech supports 400+ voices across 140+ languages. Here we use the default `en-US-JennyNeural` voice, but you can choose any voice from the [voice gallery](https://speech.microsoft.com/portal/voicegallery).

In [ ]:
%pip install azure-cognitiveservices-speech python-dotenv --quiet

In [ ]:
import os
import azure.cognitiveservices.speech as speechsdk
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

speech_key = os.environ["AZURE_AI_KEY"]
speech_region = os.environ["AZURE_AI_REGION"]

speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)
speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"

print("Speech config ready.")

In [ ]:
# Synthesize text and save the output to a WAV file
OUTPUT_FILE = "tts_output.wav"
TEXT_TO_SPEAK = "Hello! This is a text to speech demonstration using Azure AI Speech. The neural voice sounds very natural."

audio_config = speechsdk.audio.AudioOutputConfig(filename=OUTPUT_FILE)
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

print(f"Synthesizing: '{TEXT_TO_SPEAK}'")
result = synthesizer.speak_text_async(TEXT_TO_SPEAK).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"\nSpeech synthesized successfully. Audio saved to: {OUTPUT_FILE}")
elif result.reason == speechsdk.ResultReason.Canceled:
    details = result.cancellation_details
    print(f"Synthesis canceled: {details.reason}")
    if details.reason == speechsdk.CancellationReason.Error:
        print(f"Error details: {details.error_details}")

In [ ]:
# Synthesize using SSML for fine-grained control over rate, pitch, and emphasis
SSML_OUTPUT_FILE = "tts_ssml_output.wav"
ssml_text = """
<speak version='1.0' xml:lang='en-US'>
  <voice name='en-US-GuyNeural'>
    <prosody rate='slow' pitch='-10%'>
      Welcome to Azure AI Speech.
    </prosody>
    <break time='500ms'/>
    <prosody rate='fast'>
      This part is spoken faster and at a higher pitch.
    </prosody>
  </voice>
</speak>
"""

ssml_audio_config = speechsdk.audio.AudioOutputConfig(filename=SSML_OUTPUT_FILE)
ssml_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=ssml_audio_config)

ssml_result = ssml_synthesizer.speak_ssml_async(ssml_text).get()

if ssml_result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"SSML synthesis complete. Audio saved to: {SSML_OUTPUT_FILE}")
elif ssml_result.reason == speechsdk.ResultReason.Canceled:
    details = ssml_result.cancellation_details
    print(f"Synthesis canceled: {details.reason}")
    if details.reason == speechsdk.CancellationReason.Error:
        print(f"Error details: {details.error_details}")